# S3 · честная базовая модель

## Оценка полёта

Окна одного полёта объединяются в одну строку проверки. На семинаре дописываются две строки внутри готового цикла агрегации.

In [ ]:
import json
from pathlib import Path

import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix
from sklearn.metrics import precision_recall_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_sau import baseline as baseline_module
from ml_sau.course_case import PHYSICAL_FEATURES, course_holdout_masks
from ml_sau.course_case import generate_course_case


def aggregate_flight_scores(window_score, target, flight_id):
    window_score = np.asarray(window_score, dtype=np.float64)
    target = np.asarray(target, dtype=np.int64)
    flight_id = np.asarray(flight_id)
    if not (len(window_score) == len(target) == len(flight_id)):
        raise ValueError("window_score, target and flight_id must be equally sized")
    if not np.isfinite(window_score).all():
        raise ValueError("window_score must be finite")

    flights = np.unique(flight_id)
    flight_target = np.empty(len(flights), dtype=np.int64)
    flight_score = np.empty(len(flights), dtype=np.float64)
    for index, flight in enumerate(flights):
        mask = flight_id == flight
        labels = np.unique(target[mask])
        if len(labels) != 1:
            raise ValueError(f"flight {flight!r} has conflicting targets")
        raise NotImplementedError("Сохраните target и максимальный score полёта")

    return flights, flight_target, flight_score


score_example = np.array([0.2, 0.8, 0.4, 0.3])
target_example = np.array([1, 1, 0, 0])
flight_example = np.array(["F2", "F2", "F1", "F1"])
ids, targets, scores = aggregate_flight_scores(
    score_example, target_example, flight_example
)
assert ids.tolist() == ["F1", "F2"]
assert targets.tolist() == [0, 1]
assert scores.tolist() == [0.4, 0.8]

## Базовая модель и Average Precision

Подготовлены одинаковые признаки и разбиение. На семинаре выполняются `fit` и получение оконных score для двух моделей.

In [ ]:
case = generate_course_case(seed=1126)
train, validation, test = course_holdout_masks(case)
feature_idx = np.arange(len(PHYSICAL_FEATURES))

dummy = DummyClassifier(strategy="prior")
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42),
)

raise NotImplementedError("Обучите обе модели и получите два window_score")

flights, flight_target, flight_score = aggregate_flight_scores(
    window_score, case.y[validation], case.flight_id[validation]
)
_, dummy_target, dummy_flight_score = aggregate_flight_scores(
    dummy_window_score, case.y[validation], case.flight_id[validation]
)
np.testing.assert_array_equal(flight_target, dummy_target)
model_ap = average_precision_score(flight_target, flight_score)
dummy_ap = average_precision_score(flight_target, dummy_flight_score)
print("Dummy AP:", dummy_ap)
print("Logistic regression AP:", model_ap)

## Выбор порога

Массивы precision, recall и thresholds уже рассчитаны. На семинаре записывается правило ограничения $recall \ge 0.85$ и разрешения совпадений.

In [ ]:
def select_threshold(target, score, min_recall):
    target = np.asarray(target, dtype=np.int64)
    score = np.asarray(score, dtype=np.float64)
    if target.ndim != 1 or score.ndim != 1 or len(target) != len(score):
        raise ValueError("target and score must be one-dimensional and equally sized")
    if not np.isfinite(score).all():
        raise ValueError("score must be finite")
    if np.unique(target).tolist() != [0, 1]:
        raise ValueError("target must contain both classes 0 and 1")
    if not 0 < min_recall <= 1:
        raise ValueError("min_recall must belong to (0, 1]")

    precision, recall, thresholds = precision_recall_curve(target, score)
    raise NotImplementedError("Выберите допустимые пороги и лучший precision")


threshold = select_threshold(flight_target, flight_score, min_recall=0.85)
prediction = flight_score >= threshold
print("threshold:", threshold)

## Анализ FP и FN

Матрица ошибок считается по полётам. На семинаре отдельно записываются условия ложной тревоги и пропуска.

In [ ]:
matrix = confusion_matrix(flight_target, prediction, labels=[0, 1]).ravel()
raise NotImplementedError("Добавьте маски FP и FN")

print("TN, FP, FN, TP:", matrix.tolist())
print("FP flights:", flights[false_positive])
print("FN flights:", flights[false_negative])

## Повторяемость результата

Два запуска используют одни функции, конфигурацию и split. На семинаре сравниваются сохранённые файлы метрик и разбиения.

In [ ]:
output_dir = Path("reports/runs/s3-baseline")
baseline_module.run_baseline(
    Path("configs/baseline.json"),
    output_dir,
    aggregate_fn=aggregate_flight_scores,
    threshold_fn=select_threshold,
)
first_metrics = (output_dir / "validation-metrics.json").read_text()
first_split = (output_dir / "split.json").read_text()
baseline_module.run_baseline(
    Path("configs/baseline.json"),
    output_dir,
    aggregate_fn=aggregate_flight_scores,
    threshold_fn=select_threshold,
)

raise NotImplementedError("Сравните два сохранённых результата")

metrics = json.loads((output_dir / "validation-metrics.json").read_text())
print(json.dumps(metrics, ensure_ascii=False, indent=2))